<a href="https://colab.research.google.com/github/Aje-dotcom/DeepTech-/blob/master/Copy_of_Ekene__Ajemba_Assignment_Week_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Applied Learning Assignment course 1
Write a query to find all the patients in the state “Fct Abuja”, ”Plateau”.
Retrieve the total number of male and female patients.
Write a query to list doctors and their specialties in state where confirmed appointments exist.


The SQL queries required to extract the requested information are:
<Br><b>​1. Patients in "FCT Abuja" and "Plateau"​</b></br>This query uses the IN operator to filter the patient records based on their geographic location.

### Mount CSV as an SQL Query Database

To query your CSV file using SQL, we can leverage `pandas` to load the data and `sqlite3` to create an in-memory database from the DataFrame. This allows you to run standard SQL queries against your data.

In [ ]:
import pandas as pd
import sqlite3

# Define the path to your CSV file
csv_file_path = '/SQL 01 Assignment_5e2df0bafefd08c788b40df78cdf14b7.csv'

# Load the entire CSV file into a pandas DataFrame without headers initially
# This allows us to manually parse the different tables within the CSV
try:
    raw_df = pd.read_csv(csv_file_path, header=None)
    print("Raw CSV loaded successfully. First 10 rows:")
    display(raw_df.head(10)) # Display more rows to identify table sections
except FileNotFoundError:
    print(f"Error: The file at {csv_file_path} was not found. Please ensure the file path is correct.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

# --- Parse raw_df into separate DataFrames for Patients, Doctors, and Appointments ---
# Assuming a structure where tables are separated by text markers and then an empty row before headers

# Find start rows for each logical table
patients_start = raw_df[raw_df.iloc[:, 0] == 'Patients Table'].index[0] if 'Patients Table' in raw_df.iloc[:, 0].values else -1
doctors_start = raw_df[raw_df.iloc[:, 0] == 'Doctors Table'].index[0] if 'Doctors Table' in raw_df.iloc[:, 0].values else -1
appointments_start = raw_df[raw_df.iloc[:, 0] == 'Appointments Table'].index[0] if 'Appointments Table' in raw_df.iloc[:, 0].values else -1

# Extract Patients DataFrame
patients_df = pd.DataFrame()
if patients_start != -1:
    patients_header_row = patients_start + 2 # Assuming header is 2 rows after 'Patients Table' marker
    # Find the end of the patients table (before the next marker or end of file)
    patients_end = doctors_start if doctors_start != -1 else appointments_start if appointments_start != -1 else len(raw_df)
    patients_df = raw_df.iloc[patients_header_row:patients_end].copy()
    patients_df.columns = patients_df.iloc[0].str.strip() # Set the first data row as column names and strip whitespace
    patients_df = patients_df[1:].reset_index(drop=True) # Remove the header row from data
    patients_df = patients_df.dropna(axis=1, how='all') # Drop columns that are entirely NaN
    patients_df = patients_df.dropna(axis=0, how='all') # Drop rows that are entirely NaN
    print("\nPatients DataFrame Head:")
    display(patients_df.head())
else:
    print("\n'Patients Table' marker not found.")

# Extract Doctors DataFrame
doctors_df = pd.DataFrame()
if doctors_start != -1:
    doctors_header_row = doctors_start + 2 # Assuming header is 2 rows after 'Doctors Table' marker
    doctors_end = appointments_start if appointments_start != -1 else len(raw_df)
    doctors_df = raw_df.iloc[doctors_header_row:doctors_end].copy()
    doctors_df.columns = doctors_df.iloc[0].str.strip()
    doctors_df = doctors_df[1:].reset_index(drop=True)
    doctors_df = doctors_df.dropna(axis=1, how='all')
    doctors_df = doctors_df.dropna(axis=0, how='all')
    print("\nDoctors DataFrame Head:")
    display(doctors_df.head())
else:
    print("\n'Doctors Table' marker not found.")

# Extract Appointments DataFrame
appointments_df = pd.DataFrame()
if appointments_start != -1:
    appointments_header_row = appointments_start + 2 # Assuming header is 2 rows after 'Appointments Table' marker
    appointments_df = raw_df.iloc[appointments_header_row:].copy()
    appointments_df.columns = appointments_df.iloc[0].str.strip()
    appointments_df = appointments_df[1:].reset_index(drop=True)
    appointments_df = appointments_df.dropna(axis=1, how='all')
    appointments_df = appointments_df.dropna(axis=0, how='all')
    print("\nAppointments DataFrame Head:")
    display(appointments_df.head())
else:
    print("\n'Appointments Table' marker not found.")

Raw CSV loaded successfully. First 10 rows:


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,Patients Table,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,PatientID,Name,Age,Gender,State
4,10001,Jonathan Aminu,34,Male,Jigawa
5,10110,Abubakar Sani,45,Male,Kano
6,10111,Kangyang Bot,27,Female,Plateau
7,11111,John Adamu,35,Male,Fct Abuja
8,12003,Kemi Adebayo,57,Female,Ogun
9,11320,Esther Ugo,28,Female,Anambra



Patients DataFrame Head:


3,PatientID,Name,Age,Gender,State
0,10001,Jonathan Aminu,34,Male,Jigawa
1,10110,Abubakar Sani,45,Male,Kano
2,10111,Kangyang Bot,27,Female,Plateau
3,11111,John Adamu,35,Male,Fct Abuja
4,12003,Kemi Adebayo,57,Female,Ogun



Doctors DataFrame Head:


16,DoctorID,Name,Speciality,State
0,32011,Dr. John Olu,Cardiology,Nasarawa
1,32013,Dr. Baker John,Neurology,Cross River
2,32014,Dr. Aminu Abdul,Orthopedics,Sokoto
3,32015,Dr. Anita Chinedu,Dermatology,Abia
4,32016,Dr. Esther Job,Ophtalmology,Ekiti



Appointments DataFrame Head:


26,AppointmentID,PatientID,DoctorID,Date,Status
0,20101,12003,32013,10/12/2024,Confirmed
1,20102,10111,32013,11/30/2024,Confirmed
2,20103,34352,32015,1/24/2025,Pending
3,20107,11111,32015,12/5/2024,Confirmed
4,20109,12003,32014,2/9/2025,Pending


In [ ]:
# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Write the DataFrames to SQLite tables
if not patients_df.empty:
    patients_df.to_sql('Patients', conn, if_exists='replace', index=False)
    print("Patients DataFrame successfully loaded into SQLite table 'Patients'.")
else:
    print("Patients DataFrame is empty, 'Patients' table not created.")

if not doctors_df.empty:
    doctors_df.to_sql('Doctors', conn, if_exists='replace', index=False)
    print("Doctors DataFrame successfully loaded into SQLite table 'Doctors'.")
else:
    print("Doctors DataFrame is empty, 'Doctors' table not created.")

if not appointments_df.empty:
    appointments_df.to_sql('Appointments', conn, if_exists='replace', index=False)
    print("Appointments DataFrame successfully loaded into SQLite table 'Appointments'.")
else:
    print("Appointments DataFrame is empty, 'Appointments' table not created.")

print("\nVerifying table schemas:")
# Example query to show schema of Patients table
try:
    patients_schema = pd.read_sql_query("PRAGMA table_info(Patients);", conn)
    print("\nPatients Table Schema:")
    display(patients_schema)
except Exception as e:
    print(f"Error getting Patients table schema: {e}")

# Example query to show schema of Doctors table
try:
    doctors_schema = pd.read_sql_query("PRAGMA table_info(Doctors);", conn)
    print("\nDoctors Table Schema:")
    display(doctors_schema)
except Exception as e:
    print(f"Error getting Doctors table schema: {e}")

# Example query to show schema of Appointments table
try:
    appointments_schema = pd.read_sql_query("PRAGMA table_info(Appointments);", conn)
    print("\nAppointments Table Schema:")
    display(appointments_schema)
except Exception as e:
    print(f"Error getting Appointments table schema: {e}")

Patients DataFrame successfully loaded into SQLite table 'Patients'.
Doctors DataFrame successfully loaded into SQLite table 'Doctors'.
Appointments DataFrame successfully loaded into SQLite table 'Appointments'.

Verifying table schemas:

Patients Table Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,0
1,1,Name,TEXT,0,None,0
2,2,Age,TEXT,0,None,0
3,3,Gender,TEXT,0,None,0
4,4,State,TEXT,0,None,0



Doctors Table Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,DoctorID,TEXT,0,None,0
1,1,Name,TEXT,0,None,0
2,2,Speciality,TEXT,0,None,0
3,3,State,TEXT,0,None,0



Appointments Table Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,AppointmentID,TEXT,0,None,0
1,1,PatientID,TEXT,0,None,0
2,2,DoctorID,TEXT,0,None,0
3,3,Date,TEXT,0,None,0
4,4,Status,TEXT,0,None,0


Now that your CSV data is accessible via SQL within the 'data' table, you can execute your SQL queries by using `pd.read_sql_query(your_sql_query, conn)`.

In [ ]:
your_sql_query = """
SELECT * FROM Patients
WHERE State IN ('Fct Abuja', 'Plateau');
"""

# Execute the query and display results
result_df = pd.read_sql_query(your_sql_query, conn)
display(result_df)

,PatientID,Name,Age,Gender,State
0,10111,Kangyang Bot,27,Female,Plateau
1,11111,John Adamu,35,Male,Fct Abuja


In [ ]:
# Use describe() to display statistical summary of the query results
display(result_df.describe(include='all'))

,PatientID,Name,Age,Gender,State
count,2,2,2,2,2
unique,2,2,2,2,2
top,10111,Kangyang Bot,27,Female,Plateau
freq,1,1,1,1,1


<b>2. Total Number of Male and Female Patients​<br></b>This query uses the COUNT aggregate function and GROUP BY to provide a demographic breakdown of your patient population

In [ ]:
# SQL query to count patients by gender
gender_query = """
SELECT Gender, COUNT(*) AS Total_Patients
FROM Patients
GROUP BY Gender;
"""

# Execute the query and display the results
gender_dist_df = pd.read_sql_query(gender_query, conn)
display(gender_dist_df)

,Gender,Total_Patients
0,Female,4
1,Male,4


​<b>3. Doctors with Confirmed Appointments</b><br>
​This query joins the Doctors and Appointments tables to identify specialists who have active, confirmed bookings. I have included DISTINCT to ensure each doctor is only listed once even if they have multiple confirmed appointments.

In [ ]:
doctor_appointments_query = """
SELECT DISTINCT d.Name, d.Speciality, d.State
FROM Doctors d
JOIN Appointments a ON d.DoctorID = a.DoctorID
WHERE a.Status = 'Confirmed';
"""

# Execute the join query and display results
confirmed_doctors_df = pd.read_sql_query(doctor_appointments_query, conn)
display(confirmed_doctors_df)

,Name,Speciality,State
0,Dr. Baker John,Neurology,Cross River
1,Dr. Anita Chinedu,Dermatology,Abia


I’ve been diving deep into the healthcare data landscape, focusing on how structured querying can streamline hospital operations and patient care. As a healthcare analyst, my focus remains on transforming raw data into clear, actionable insights.
<P>Key Highlights from (Course 1) week 2 assignment:</h4>
<h4>Geographic Insights: </h4>Leveraged SQL to filter patient populations across specific regions like Fct Abuja and Plateau, helping to identify regional health trends and resource needs.
<h4>Demographic Analysis: </h4>Conducted a gender-based census of our patient database. Understanding our demographic split is a critical first step toward personalized care and targeted health programs.
<h4>Operational Efficiency:</h4> Mapped confirmed appointments back to specific provider specialties. This allows us to see exactly where our "confirmed" demand lies, ensuring our specialists are utilized effectively.<p>
The Bigger Picture:</p>
Data isn't just rows in a table; in healthcare, it represents a patient’s journey. By mastering these relational queries, we aren't just "managing a system"—we are ensuring that the right doctors are connected to the right patients at the right time.